# Nível 1 — Parte A: Triagem de Operações PLD

Desafio técnico de estágio em IA para PLD (Prevenção à Lavagem de Dinheiro) — banco fictício.

**Objetivo desta parte:** carregar as operações de `dados/dados_nivel_1.json`, investigar a qualidade dos dados *antes* de qualquer tratamento, corrigir os problemas encontrados de forma documentada, normalizar valores para BRL, gerar agregações básicas e implementar duas regras de triagem (Regra 1 — Fracionamento; Regra 2 — Valor Atípico).

**Regra de ouro:** toda soma, contagem, mediana e comparação numérica é feita em pandas/Python puro — nenhum LLM é usado nesta etapa.

> Assume-se que o notebook é executado a partir da pasta `nivel_1/` (diretório padrão do Jupyter ao abrir este arquivo), por isso os caminhos de dados usam `../dados/`.

## 1. Carregamento dos dados

O JSON tem dois níveis: a taxa de câmbio (`taxa_cambio_usd_brl`) no topo e a lista de operações em `operacoes`. Carregamos o arquivo bruto com `json.load` para não perder a taxa de câmbio, e construímos o DataFrame apenas a partir da lista de operações.

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_PATH = Path("..") / "dados" / "dados_nivel_1.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    dados_brutos = json.load(f)

taxa_cambio_usd_brl: float = dados_brutos["taxa_cambio_usd_brl"]
df = pd.DataFrame(dados_brutos["operacoes"])

print(f"Taxa de câmbio USD/BRL: {taxa_cambio_usd_brl}")
print(f"Formato do DataFrame (linhas, colunas): {df.shape}")
df.head(10)

## 2. Investigação de qualidade dos dados (antes de tratar)

Antes de corrigir qualquer coisa, precisamos saber exatamente o que está errado. Verificamos, nesta ordem:

1. **Duplicatas por `id`** — se o mesmo `id` aparece mais de uma vez, isso infla contagens e somas de forma silenciosa (grave para regras baseadas em soma/contagem, como a Regra 1).
2. **Valores nulos por coluna** — em especial `data`, que é usada para agrupar operações por dia.
3. **Tipos e valores únicos de `canal`, `tipo` e `moeda`** — para detectar inconsistências de categoria (typos, capitalização, categorias inesperadas) antes de usá-las em agregações.
4. **Outras anomalias** — valores não positivos, `dtypes` inesperados e contagem de clientes distintos.

In [ ]:
# 2.1 Duplicatas por id
qtd_ids_duplicados = df.duplicated(subset="id").sum()
print(f"Linhas com id duplicado (excluindo a 1ª ocorrência): {qtd_ids_duplicados}")

print("\nTodas as linhas envolvidas em duplicidade de id:")
display(df[df.duplicated(subset="id", keep=False)].sort_values("id"))

# 2.2 Valores nulos por coluna
print("\nValores nulos por coluna:")
print(df.isna().sum())

# 2.3 Tipos e valores únicos de colunas categóricas
print("\ndtypes:")
print(df.dtypes)

print("\nValores únicos de 'canal':", sorted(df["canal"].dropna().unique().tolist()))
print("Valores únicos de 'tipo':", sorted(df["tipo"].dropna().unique().tolist()))
print("Valores únicos de 'moeda':", sorted(df["moeda"].dropna().unique().tolist()))

# 2.4 Outras anomalias
print("\nEstatísticas de 'valor':")
print(df["valor"].describe())

print("\nOperações com valor <= 0:", int((df["valor"] <= 0).sum()))
print(f"Clientes distintos ({df['cliente_id'].nunique()}):", sorted(df["cliente_id"].unique().tolist()))

### Achados da investigação

- **1 `id` duplicado:** `OP-0007` (cliente `CLI-A-3`, 2026-03-05) aparece duas vezes com dados idênticos. Se não for removido antes de agrupar, infla artificialmente a soma diária desse cliente e poderia gerar um falso positivo na Regra 1.
- **1 `data` nula:** `OP-0017` (cliente `CLI-A-5`), com a observação `"data nao capturada pelo sistema"` — indica falha de captura, não ausência real de operação.
- **`canal`, `tipo` e `moeda` estão consistentes:** sem typos ou variação de capitalização. `canal` ∈ {pix, ted, boleto, cartao, especie}; `tipo` ∈ {transferencia_enviada, transferencia_recebida, pagamento, deposito}; `moeda` ∈ {BRL, USD}.
- **Sem valores não positivos** em `valor`.
- **6 clientes distintos**, como esperado (`CLI-A-1` a `CLI-A-6`).

Esses dois problemas (duplicata de `id` e `data` nula) são os únicos defeitos estruturais nos dados e serão tratados na próxima seção, cada um com sua justificativa.

## 3. Tratamento dos problemas encontrados

**3.1 Deduplicação por `id`.** Mantemos a primeira ocorrência (`keep="first"`) e descartamos as demais. Como as linhas duplicadas encontradas são idênticas em todos os campos, não há ambiguidade sobre qual "versão" manter — a decisão relevante é apenas não contar a mesma operação duas vezes nas somas e contagens.

**3.2 `data` nula.** Em vez de descartar a operação (ela é uma movimentação financeira real e precisa continuar auditável) ou inventar uma data, preenchemos o campo com o marcador textual `"DATA_PENDENTE"` e criamos a coluna booleana `ALERTA_DATA_AUSENTE` para tornar o problema visível e rastreável em qualquer relatório futuro — sem que ele se disfarce de dado válido.

**3.3 Regra de uso do marcador `DATA_PENDENTE`.** Uma data desconhecida não pode ser comparada com outras datas para saber se ocorreu "no mesmo dia" — por isso, operações com `DATA_PENDENTE`:
- são **excluídas do agrupamento diário da Regra 1** (Fracionamento), já que essa regra depende de saber se várias operações aconteceram na mesma data;
- **permanecem ativas** nas agregações gerais (volume por cliente, contagem por canal) e na Regra 2 (Valor Atípico), pois essas análises não dependem da data — excluir a operação delas esconderia volume financeiro real do cliente.

In [ ]:
DATA_PENDENTE = "DATA_PENDENTE"

linhas_antes = len(df)

# 3.1 Deduplicação por id, preservando a primeira ocorrência
df = df.drop_duplicates(subset="id", keep="first").reset_index(drop=True)

print(f"Linhas antes da deduplicação: {linhas_antes}")
print(f"Linhas depois da deduplicação: {len(df)}")

# 3.2 Flag de alerta ANTES de preencher a data nula, para não perder a informação de ausência
df["ALERTA_DATA_AUSENTE"] = df["data"].isna()

# 3.3 Preenchimento do marcador de data pendente
df["data"] = df["data"].fillna(DATA_PENDENTE)

print(f"\nOperações com ALERTA_DATA_AUSENTE: {int(df['ALERTA_DATA_AUSENTE'].sum())}")
display(df[df["ALERTA_DATA_AUSENTE"]])

## 4. Normalização de valores para BRL

Todas as regras de triagem comparam valores em uma única unidade monetária. Como `moeda` só assume `BRL` ou `USD` (confirmado na investigação), criamos `valor_brl`: quando `moeda == "USD"`, convertemos multiplicando por `taxa_cambio_usd_brl`; caso contrário, o valor já está em BRL e é mantido como está. A partir daqui, todas as agregações e regras usam `valor_brl`, nunca `valor`.

In [ ]:
df["valor_brl"] = df["valor"].where(df["moeda"] != "USD", df["valor"] * taxa_cambio_usd_brl)

print("Operações em USD convertidas:")
display(df.loc[df["moeda"] == "USD", ["id", "cliente_id", "moeda", "valor", "valor_brl"]])

df[["id", "cliente_id", "moeda", "valor", "valor_brl"]].head(10)